# ⚠️ Required First Step — Enable GPU

1. Click **Runtime** in the menu above
2. Click **Change runtime type**
3. Set **Hardware accelerator** to **T4 GPU** (free) or **A100**
4. Click **Save**

Then click **Runtime → Run all** (Ctrl+F9 / Cmd+F9). Takes ~3 minutes.

---

# TunedAI Labs — Causal Reasoning Benchmark

Runs both models against the real **CLadder benchmark** — the same 10,112-question test used to establish the 96.96% score.

| Model | CLadder Score |
|---|---|
| Base Qwen 2.5-7B | ~62% |
| **TunedAI Labs Causal Model** | **96.96%** |

Questions use fictional variable names (yupt, jyka, kwox, etc.) — models cannot recall answers from pretraining. Requires actual causal reasoning.

This notebook runs a **500-question stratified sample** (~30 min on T4). Results should land near the published scores.

---

## Step 1 — Install packages

In [ ]:
import subprocess, sys, importlib

pkgs = ['transformers', 'peft', 'accelerate', 'bitsandbytes', 'openai', 'anthropic']
import_names = {'transformers': 'transformers', 'peft': 'peft', 'accelerate': 'accelerate',
                'bitsandbytes': 'bitsandbytes', 'openai': 'openai', 'anthropic': 'anthropic'}

for pkg in pkgs:
    try:
        importlib.import_module(import_names[pkg])
        print(f"  {pkg} already installed ✓")
    except ImportError:
        print(f"  installing {pkg}...", end=" ", flush=True)
        r = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-q', pkg],
            capture_output=True, text=True, timeout=180)
        if r.returncode == 0:
            print("✓")
        else:
            print(f"FAILED\n{r.stderr[-200:]}")

print("\n✓ All packages ready.")

---
## Step 2 — API keys (optional)

Leave blank to skip GPT-4o and Claude columns — the local model comparison still runs.

In [ ]:
OPENAI_API_KEY    = ""   # paste OpenAI key here (optional)
ANTHROPIC_API_KEY = ""   # paste Anthropic key here (optional)

print("OpenAI key:",    "set" if OPENAI_API_KEY    else "not set (GPT-4o column will be skipped)")
print("Anthropic key:", "set" if ANTHROPIC_API_KEY else "not set (Claude column will be skipped)")

---
## Step 3 — Load models

Downloads Qwen 2.5-7B-Instruct + TunedAI Labs LoRA adapter from HuggingFace.
Takes ~90 seconds on T4.

In [ ]:
import torch, os
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, logging
from peft import PeftModel
import openai, anthropic

logging.set_verbosity_info()  # show download progress

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Go to Runtime → Change runtime type → set T4 GPU.")
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {torch.cuda.get_device_name(0)}  |  VRAM: {vram_gb:.0f} GB")

BASE_MODEL   = "Qwen/Qwen2.5-7B-Instruct"
ADAPTER_REPO = "tunedailabs/causal-reasoning-qwen-7b"

print("\nLoading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
print("✓ Tokenizer ready")

if vram_gb >= 20:
    print("\nLoading base model float16 (downloading ~14GB, takes 3-5 min)...")
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, torch_dtype=torch.float16, device_map="cuda:0", trust_remote_code=True)
else:
    print("\nLoading base model 8-bit (downloading ~14GB, takes 3-5 min)...")
    bnb_config = BitsAndBytesConfig(load_in_8bit=True)
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, quantization_config=bnb_config, device_map="auto", trust_remote_code=True)
print("✓ Base model ready")

print("\nLoading TunedAI Labs adapter...")
tuned_model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)
tuned_model.eval()
print("✓ Adapter ready")

oai_client = openai.OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None
ant_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY) if ANTHROPIC_API_KEY else None

logging.set_verbosity_warning()  # quiet again for inference
print("\n✓ All models ready.")

---
## Step 4 — Verify adapter integrity

Checks SHA256 of the downloaded adapter against the known-good v2_e1 hash (96.96% CLadder).

In [ ]:
import hashlib, os, glob

# SHA256 of the correct v2_e1 adapter (96.96% CLadder, 9805/10112 correct)
KNOWN_SHA256 = "e0967ac8460870eab2f114379c4d4e9640934f944e294e115eec3f6bc67eba13"

# Find the cached adapter file (HuggingFace downloads to ~/.cache/huggingface)
cache_pattern = os.path.expanduser(
    "~/.cache/huggingface/hub/models--tunedailabs--causal-reasoning-qwen-7b/**/*.safetensors")
candidates = [f for f in glob.glob(cache_pattern, recursive=True)
              if not f.endswith(".incomplete")]

if not candidates:
    print("⚠ Adapter not found in cache — run Step 3 first.")
else:
    adapter_file = candidates[0]
    print(f"Verifying: {os.path.basename(adapter_file)} ({os.path.getsize(adapter_file)/1e6:.0f} MB)")
    print("Computing SHA256...", end=" ", flush=True)

    sha = hashlib.sha256()
    with open(adapter_file, "rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            sha.update(chunk)
    computed = sha.hexdigest()

    if computed == KNOWN_SHA256:
        print("✓")
        print(f"\n✓ SHA256 MATCH — confirmed v2_e1 adapter (96.96% CLadder)")
        print(f"  {computed}")
    else:
        print("✗")
        print(f"\n✗ SHA256 MISMATCH — wrong adapter loaded")
        print(f"  Expected: {KNOWN_SHA256}")
        print(f"  Got:      {computed}")
        print("\nFix: Runtime → Disconnect and delete runtime, then start over.")
        raise RuntimeError("Wrong adapter. Results would be invalid.")

---
## Step 5 — Run benchmark

Two modes — toggle `USE_CLADDER` below:

- `USE_CLADDER = False` ← **default** — generates fresh questions the model has never seen. Correct answers computed from the numbers. Proves genuine causal reasoning.
- `USE_CLADDER = True` — uses the real CLadder dataset (model was trained on this distribution).

Takes ~3 min for 20 questions, ~20 min for 200.

In [ ]:
import json, os, random, urllib.request, torch

# ── CONFIG ────────────────────────────────────────────────────────────────────
N_QUESTIONS  = 200
RANDOM_SEED  = 42
USE_CLADDER  = False       # False = fresh generated (never seen by model)
                           # True  = real CLadder dataset
RESULTS_FILE = "/content/results.json"
# ─────────────────────────────────────────────────────────────────────────────

SYSTEM = "You are a causal reasoning expert. Answer YES or NO only."
MODE   = "cladder" if USE_CLADDER else "generated"

VARS = [
    "florp","wumf","ziff","quunt","brent","mirk","drex","veld","plox","snuv",
    "trake","yulf","grizz","blont","spuv","kornt","whelp","frix","zolt","blumb",
    "krenz","vusp","torb","wenf","glurt","plix","snorv","bwomp","cruft","zilth",
    "greff","polm","dweez","slunx","fruvv","kwelp","throx","blimp","snurt","grolt"
]

def gen_fresh(seed):
    rng = random.Random(seed)
    X, Y = rng.sample(VARS, 2)
    qt = rng.choice(['r1_more','r1_less','r1_overall','r2_up','r2_down','r3'])
    def pct(v): return f"{round(v*100)}%"

    if qt in ('r1_more','r1_less'):
        px=round(rng.uniform(0.2,0.8),2); py1=round(rng.uniform(0.1,0.9),2); py0=round(rng.uniform(0.1,0.9),2)
        while abs(py1-py0)<0.12: py1=round(rng.uniform(0.1,0.9),2)
        gi=(f"The overall probability of {X} is {pct(px)}. "
            f"For those who are not {X}, the probability of {Y} is {pct(py0)}. "
            f"For those who are {X}, the probability of {Y} is {pct(py1)}.")
        if qt=='r1_more': return dict(given=gi, q=f"Is {Y} more likely when observing {X}?", a='yes' if py1>py0 else 'no', rung=1)
        else:             return dict(given=gi, q=f"Is {Y} less likely when observing {X}?",  a='yes' if py1<py0 else 'no', rung=1)

    elif qt=='r1_overall':
        px=round(rng.uniform(0.2,0.8),2); py1=round(rng.uniform(0.1,0.9),2); py0=round(rng.uniform(0.1,0.9),2)
        py=px*py1+(1-px)*py0
        gi=(f"The overall probability of {X} is {pct(px)}. "
            f"The probability of not {X} and {Y} is {pct(round((1-px)*py0,2))}. "
            f"The probability of {X} and {Y} is {pct(round(px*py1,2))}.")
        d=rng.choice(['more','less'])
        if d=='more': return dict(given=gi, q=f"Is {Y} more likely than not {Y} overall?", a='yes' if py>0.5 else 'no', rung=1)
        else:         return dict(given=gi, q=f"Is {Y} less likely than not {Y} overall?",  a='yes' if py<0.5 else 'no', rung=1)

    elif qt in ('r2_up','r2_down'):
        py1=round(rng.uniform(0.1,0.9),2); py0=round(rng.uniform(0.1,0.9),2)
        while abs(py1-py0)<0.12: py1=round(rng.uniform(0.1,0.9),2)
        gi=(f"For those who are not {X}, the probability of {Y} is {pct(py0)}. "
            f"For those who are {X}, the probability of {Y} is {pct(py1)}.")
        if qt=='r2_up': return dict(given=gi, q=f"Will {X} increase the chance of {Y}?",  a='yes' if py1>py0 else 'no', rung=2)
        else:           return dict(given=gi, q=f"Will {X} decrease the chance of {Y}?",  a='yes' if py1<py0 else 'no', rung=2)

    else:
        py1=round(rng.uniform(0.1,0.9),2); py0=round(rng.uniform(0.1,0.9),2)
        while abs(py1-py0)<0.12: py1=round(rng.uniform(0.1,0.9),2)
        gi=(f"For those who are not {X}, the probability of {Y} is {pct(py0)}. "
            f"For those who are {X}, the probability of {Y} is {pct(py1)}.")
        d=rng.choice(['more','less'])
        if d=='more': return dict(given=gi, q=f"For those who are {X}, would it be more likely to see {Y} if the individual was not {X}?", a='yes' if py0>py1 else 'no', rung=3)
        else:         return dict(given=gi, q=f"For those who are {X}, would it be less likely to see {Y} if the individual was not {X}?", a='yes' if py0<py1 else 'no', rung=3)

# ── Build question list ───────────────────────────────────────────────────────
if USE_CLADDER:
    path="/content/cladder-v1-questions.json"
    if not os.path.exists(path):
        print("Downloading CLadder...", end=" ", flush=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/causalNLP/cladder/main/data/cladder-v1-questions.json", path)
        print("✓")
    with open(path) as f: all_q=json.load(f)
    random.seed(RANDOM_SEED)
    buckets={}
    for q in all_q:
        k=(q.get('meta',{}).get('rung',0), q.get('answer','').lower()); buckets.setdefault(k,[]).append(q)
    qs=[]
    per=max(1, N_QUESTIONS//len(buckets))
    for k in sorted(buckets): qs+=random.sample(buckets[k], min(per,len(buckets[k])))
    random.shuffle(qs); qs=qs[:N_QUESTIONS]
    questions=[dict(id=q.get('question_id',str(i)),
                    given=q.get('given_info','') if isinstance(q.get('given_info',''),str) else ' '.join(f"{k}: {v}" for k,v in q['given_info'].items()),
                    q=q['question'], a=q['answer'].lower().strip(),
                    rung=q.get('meta',{}).get('rung','?')) for i,q in enumerate(qs)]
    print(f"CLadder: {len(questions)} real benchmark questions")
else:
    questions=[{**gen_fresh(RANDOM_SEED+i), 'id': f"gen_{RANDOM_SEED}_{i}"} for i in range(N_QUESTIONS)]
    print(f"Generated {len(questions)} fresh questions — model has never seen these")
    print("Answers computed mathematically from the given probabilities.")

# ── Load saved results (only if same mode) ────────────────────────────────────
saved_results = {}
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE) as f: saved = json.load(f)
    if saved.get("mode") == MODE:
        saved_results = saved.get("data", {})
        print(f"Resuming — {len(saved_results)} of {len(questions)} done.")
    else:
        print(f"Mode changed ({saved.get('mode')} → {MODE}) — starting fresh.")

# ── Run ───────────────────────────────────────────────────────────────────────
def _yn(text):
    t=text.strip().lower(); w=t.split(); f=w[0].rstrip(".,!:") if w else ""
    if f in ("yes","no"): return f
    if t.startswith("yes"): return "yes"
    if t.startswith("no"):  return "no"
    s=t[:60]
    if "yes" in s and "no" not in s: return "yes"
    if "no"  in s and "yes" not in s: return "no"
    return "?"

def ask(question, use_adapter=True):
    if use_adapter: tuned_model.enable_adapter_layers()
    else:           tuned_model.disable_adapter_layers()
    msgs=[{"role":"system","content":SYSTEM},{"role":"user","content":question}]
    text=tokenizer.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True)
    inp=tokenizer(text,return_tensors="pt").to("cuda")
    with torch.no_grad(): out=tuned_model.generate(**inp,max_new_tokens=10,do_sample=False)
    return tokenizer.decode(out[0][inp.input_ids.shape[1]:],skip_special_tokens=True).strip()

results = dict(saved_results)
b_right=t_right=done=0

for i, q in enumerate(questions):
    key=q['id']; full_q=f"{q['given']}\n{q['q']}" if q['given'] else q['q']
    expected=q['a']; rung=q['rung']

    if key in results:
        r=results[key]
        b_right+=(r['b']==expected); t_right+=(r['t']==expected); done+=1
        print(f"[Q{i+1} L{rung} — Base {'✓' if r['b']==expected else '✗'} · TunedAI {'✓' if r['t']==expected else '✗'}]")
        continue

    b=ask(full_q,False); t=ask(full_q,True)
    by=_yn(b); ty=_yn(t); bok=by==expected; tok=ty==expected
    b_right+=bok; t_right+=tok; done+=1
    flag=" ★" if tok and not bok else (" ↓" if bok and not tok else "")
    print(f"Q{i+1:>3} L{rung} | exp:{expected.upper()} "
          f"base:{by.upper()} {'✓' if bok else '✗'} tuned:{ty.upper()} {'✓' if tok else '✗'}{flag}  "
          f"[Base {b_right}/{done}={100*b_right/done:.0f}%  TunedAI {t_right}/{done}={100*t_right/done:.0f}%]")
    results[key]={"b":by,"t":ty,"ba":b,"ta":t,"a":expected,"rung":rung}
    with open(RESULTS_FILE,"w") as f: json.dump({"mode":MODE,"data":results},f,indent=2)

label = "CLadder benchmark" if USE_CLADDER else "Fresh generated (never seen by model)"
print(f"\n{'='*60}\n{label} — {done} questions")
print(f"  Base Qwen 2.5-7B  : {b_right}/{done} = {100*b_right/done:.1f}%")
print(f"  TunedAI Labs ★    : {t_right}/{done} = {100*t_right/done:.1f}%")
print(f"  Gap               : {100*(t_right-b_right)/done:+.1f} pp")
if not USE_CLADDER:
    print(f"\n  ✓ Answers computed from math — not from training data.")
print(f"{'='*60}")

---
## Results

In [ ]:
import json, os
from collections import defaultdict

RESULTS_FILE = "/content/results.json"

if not os.path.exists(RESULTS_FILE):
    print("No results yet — run the benchmark cell above first.")
else:
    with open(RESULTS_FILE) as f: saved = json.load(f)
    mode = saved.get("mode", "unknown")
    res  = saved.get("data", {})
    n    = len(res)

    if n == 0:
        print("Results file is empty — run the benchmark cell first.")
    else:
        b_correct = t_correct = 0
        level_stats = defaultdict(lambda: {"b":0,"t":0,"n":0})

        for r in res.values():
            expected = r.get("a","?")
            lvl      = r.get("rung","?")
            bok = r["b"] == expected
            tok = r["t"] == expected
            b_correct += bok
            t_correct += tok
            level_stats[lvl]["n"] += 1
            level_stats[lvl]["b"] += bok
            level_stats[lvl]["t"] += tok

        label = "CLadder benchmark" if mode == "cladder" else "Fresh generated (never seen by model)"
        SEP = "="*60
        print(SEP)
        print(f"{label} — {n} questions")
        print(SEP)
        print(f"  Base Qwen 2.5-7B  : {b_correct:3d}/{n} = {100*b_correct/n:.1f}%")
        print(f"  TunedAI Labs ★    : {t_correct:3d}/{n} = {100*t_correct/n:.1f}%")
        print(f"  Gap               : {100*(t_correct-b_correct)/n:+.1f} pp")
        print(SEP)
        print("\nBy reasoning level:")
        for lvl in sorted(level_stats.keys()):
            s = level_stats[lvl]
            print(f"  Level {lvl}: Base {s['b']}/{s['n']}={100*s['b']/s['n']:.0f}%  "
                  f"TunedAI {s['t']}/{s['n']}={100*s['t']/s['n']:.0f}%")
        if mode != "cladder":
            print(f"\n  ✓ Answers computed from math — not from training data.")

---
## Quick Demo — 20 Example Questions

Twenty CLadder-format questions (4 Level-1, 6 Level-2, 10 Level-3). Correct answers computed from the numbers — the model cannot look these up.

In [ ]:
DEMO_QUESTIONS = [
    # ── Level 1: Association (4 questions) ───────────────────────────────────
    dict(level=1,
         given="The overall probability of yupt is 60%. "
               "For those who are not yupt, the probability of kwox is 20%. "
               "For those who are yupt, the probability of kwox is 75%.",
         q="Is kwox more likely when observing yupt?",
         a="yes"),   # 75 > 20

    dict(level=1,
         given="The overall probability of jyka is 45%. "
               "For those who are not jyka, the probability of glimx is 70%. "
               "For those who are jyka, the probability of glimx is 30%.",
         q="Is glimx less likely when observing jyka?",
         a="yes"),   # 30 < 70

    dict(level=1,
         given="The overall probability of zory is 35%. "
               "For those who are not zory, the probability of flam is 80%. "
               "For those who are zory, the probability of flam is 55%.",
         q="Is flam more likely when observing zory?",
         a="no"),    # 55 < 80 — tricky trap question

    dict(level=1,
         given="The overall probability of kwox is 50%. "
               "For those who are not kwox, the probability of yupt is 25%. "
               "For those who are kwox, the probability of yupt is 68%.",
         q="Is yupt more likely when observing kwox?",
         a="yes"),   # 68 > 25

    # ── Level 2: Intervention (6 questions) ──────────────────────────────────
    dict(level=2,
         given="For those who are not yupt, the probability of kwox is 15%. "
               "For those who are yupt, the probability of kwox is 82%.",
         q="Will yupt increase the chance of kwox?",
         a="yes"),   # 82 > 15

    dict(level=2,
         given="For those who are not jyka, the probability of glimx is 65%. "
               "For those who are jyka, the probability of glimx is 28%.",
         q="Will jyka decrease the chance of glimx?",
         a="yes"),   # 28 < 65

    dict(level=2,
         given="For those who are not wroe, the probability of qwiu is 40%. "
               "For those who are wroe, the probability of qwiu is 73%.",
         q="Will wroe decrease the chance of qwiu?",
         a="no"),    # 73 > 40 — it increases, not decreases

    dict(level=2,
         given="For those who are not fibt, the probability of pexu is 62%. "
               "For those who are fibt, the probability of pexu is 38%.",
         q="Will fibt increase the chance of pexu?",
         a="no"),    # 38 < 62 — it decreases, not increases

    dict(level=2,
         given="For those who are not swoy, the probability of rukz is 30%. "
               "For those who are swoy, the probability of rukz is 70%.",
         q="Will swoy decrease the chance of rukz?",
         a="no"),    # 70 > 30 — it increases

    dict(level=2,
         given="For those who are not zory, the probability of flam is 75%. "
               "For those who are zory, the probability of flam is 25%.",
         q="Will zory increase the chance of flam?",
         a="no"),    # 25 < 75 — it decreases

    # ── Level 3: Counterfactual (10 questions) ────────────────────────────────
    dict(level=3,
         given="For those who are not yupt, the probability of kwox is 20%. "
               "For those who are yupt, the probability of kwox is 80%.",
         q="For those who are yupt, would it be less likely to see kwox if the individual was not yupt?",
         a="yes"),   # P(kwox|not yupt)=20 < P(kwox|yupt)=80 → yes, less likely

    dict(level=3,
         given="For those who are not jyka, the probability of glimx is 72%. "
               "For those who are jyka, the probability of glimx is 31%.",
         q="For those who are jyka, would it be more likely to see glimx if the individual was not jyka?",
         a="yes"),   # P(glimx|not jyka)=72 > P(glimx|jyka)=31 → yes, more likely

    dict(level=3,
         given="For those who are not rukz, the probability of swoy is 45%. "
               "For those who are rukz, the probability of swoy is 78%.",
         q="For those who are rukz, would it be more likely to see swoy if the individual was not rukz?",
         a="no"),    # P(swoy|not rukz)=45 < P(swoy|rukz)=78 → no, less likely

    dict(level=3,
         given="For those who are not pexu, the probability of rukz is 15%. "
               "For those who are pexu, the probability of rukz is 85%.",
         q="For those who are pexu, would it be less likely to see rukz if the individual was not pexu?",
         a="yes"),   # P(rukz|not pexu)=15 < P(rukz|pexu)=85 → yes, less likely

    dict(level=3,
         given="For those who are not wroe, the probability of fibt is 70%. "
               "For those who are wroe, the probability of fibt is 25%.",
         q="For those who are wroe, would it be more likely to see fibt if the individual was not wroe?",
         a="yes"),   # P(fibt|not wroe)=70 > P(fibt|wroe)=25 → yes, more likely

    dict(level=3,
         given="For those who are not qwiu, the probability of zory is 80%. "
               "For those who are qwiu, the probability of zory is 30%.",
         q="For those who are qwiu, would it be less likely to see zory if the individual was not qwiu?",
         a="no"),    # P(zory|not qwiu)=80 > P(zory|qwiu)=30 → no, MORE likely (not less)

    dict(level=3,
         given="For those who are not glimx, the probability of yupt is 55%. "
               "For those who are glimx, the probability of yupt is 90%.",
         q="For those who are glimx, would it be more likely to see yupt if the individual was not glimx?",
         a="no"),    # P(yupt|not glimx)=55 < P(yupt|glimx)=90 → no, less likely

    dict(level=3,
         given="For those who are not kwox, the probability of jyka is 20%. "
               "For those who are kwox, the probability of jyka is 75%.",
         q="For those who are kwox, would it be less likely to see jyka if the individual was not kwox?",
         a="yes"),   # P(jyka|not kwox)=20 < P(jyka|kwox)=75 → yes, less likely

    dict(level=3,
         given="For those who are not flam, the probability of zory is 65%. "
               "For those who are flam, the probability of zory is 15%.",
         q="For those who are flam, would it be more likely to see zory if the individual was not flam?",
         a="yes"),   # P(zory|not flam)=65 > P(zory|flam)=15 → yes, more likely

    dict(level=3,
         given="For those who are not pexu, the probability of fibt is 40%. "
               "For those who are pexu, the probability of fibt is 85%.",
         q="For those who are pexu, would it be more likely to see fibt if the individual was not pexu?",
         a="no"),    # P(fibt|not pexu)=40 < P(fibt|pexu)=85 → no, less likely
]

print("Running 20 demo questions (4 L1 · 6 L2 · 10 L3)...\n")
print(f"{'Q':<4} {'Lvl':<5} {'Exp':<6} {'Base':<8} {'TunedAI':<10} {'Result'}")
print("-"*58)

b_right = t_right = 0
for i, dq in enumerate(DEMO_QUESTIONS):
    full_q = f"{dq['given']}\n{dq['q']}"
    ba = ask(full_q, use_adapter=False)
    ta = ask(full_q, use_adapter=True)
    by = _yn(ba); ty = _yn(ta); exp = dq['a']
    bok = by == exp; tok = ty == exp
    b_right += bok; t_right += tok
    result = "★ TunedAI wins" if tok and not bok else ("✓ both right" if bok and tok else ("✗ both wrong" if not bok and not tok else "↓ base wins"))
    print(f"Q{i+1:<3} L{dq['level']:<4} {exp.upper():<6} {by.upper():<8} {ty.upper():<10} {result}")

n = len(DEMO_QUESTIONS)
print(f"\n{'='*58}")
print(f"Base Qwen 2.5-7B  : {b_right}/{n} = {100*b_right/n:.0f}%")
print(f"TunedAI Labs ★    : {t_right}/{n} = {100*t_right/n:.0f}%")
print(f"Gap               : {100*(t_right-b_right)/n:+.0f} pp")
print(f"{'='*58}")

---
## Build Your Own Question

Fill in the template below and run the cell. The question format matches the CLadder benchmark exactly.

**Three question types:**

**Level 1 (Association)** — observational
> Given: prob of X = P%, prob of Y given not-X = A%, prob of Y given X = B%
> Question: Is Y more/less likely when observing X?
> Answer: yes if B > A (more likely) or B < A (less likely)

**Level 2 (Intervention)** — causal
> Given: prob of Y given not-X = A%, prob of Y given X = B%
> Question: Will X increase/decrease the chance of Y?
> Answer: yes if B > A (increase) or B < A (decrease)

**Level 3 (Counterfactual)**
> Given: prob of Y given not-X = A%, prob of Y given X = B%
> Question: For those who ARE X, would Y be more/less likely if they were NOT X?
> Answer: yes if A > B (more likely) or A < B (less likely)

In [ ]:
# ── Fill in your own question ─────────────────────────────────────────────────
X = "yupt"    # first variable name (any word)
Y = "kwox"    # second variable name (any word)

P_X      = 0.55   # probability of X  (Level 1 only — ignored for L2/L3)
P_Y_notX = 0.20   # probability of Y given NOT X
P_Y_X    = 0.78   # probability of Y given X

LEVEL    = 2      # 1 = association, 2 = intervention, 3 = counterfactual

# Question direction: "increase"/"decrease" (L2), "more"/"less" (L1/L3)
DIRECTION = "increase"   # for L2: "increase" or "decrease"
                         # for L1: "more" or "less"
                         # for L3: "more" or "less"
# ─────────────────────────────────────────────────────────────────────────────

def pct(v): return f"{round(v*100)}%"

if LEVEL == 1:
    given = (f"The overall probability of {X} is {pct(P_X)}. "
             f"For those who are not {X}, the probability of {Y} is {pct(P_Y_notX)}. "
             f"For those who are {X}, the probability of {Y} is {pct(P_Y_X)}.")
    question = f"Is {Y} {DIRECTION} likely when observing {X}?"
    correct = "yes" if (DIRECTION == "more" and P_Y_X > P_Y_notX) or (DIRECTION == "less" and P_Y_X < P_Y_notX) else "no"

elif LEVEL == 2:
    given = (f"For those who are not {X}, the probability of {Y} is {pct(P_Y_notX)}. "
             f"For those who are {X}, the probability of {Y} is {pct(P_Y_X)}.")
    question = f"Will {X} {DIRECTION} the chance of {Y}?"
    correct = "yes" if (DIRECTION == "increase" and P_Y_X > P_Y_notX) or (DIRECTION == "decrease" and P_Y_X < P_Y_notX) else "no"

else:  # Level 3
    given = (f"For those who are not {X}, the probability of {Y} is {pct(P_Y_notX)}. "
             f"For those who are {X}, the probability of {Y} is {pct(P_Y_X)}.")
    question = f"For those who are {X}, would it be {DIRECTION} likely to see {Y} if the individual was not {X}?"
    correct = "yes" if (DIRECTION == "more" and P_Y_notX > P_Y_X) or (DIRECTION == "less" and P_Y_notX < P_Y_X) else "no"

full_q = f"{given}\n{question}"
print(f"Question (Level {LEVEL}):")
print(f"  {given}")
print(f"  {question}")
print(f"\nCorrect answer (from math): {correct.upper()}\n")

base_ans  = ask(full_q, use_adapter=False)
tuned_ans = ask(full_q, use_adapter=True)
base_yn   = _yn(base_ans)
tuned_yn  = _yn(tuned_ans)

print(f"Base Qwen 2.5-7B  : {base_yn.upper()}  {'✓' if base_yn == correct else '✗'}")
print(f"TunedAI Labs ★    : {tuned_yn.upper()}  {'✓' if tuned_yn == correct else '✗'}")

---
## Share Your Results

Open a [GitHub Issue](https://github.com/tunedailabs/tunedailabs/issues/new) and paste what you saw.

**TunedAI Labs** — We fine-tune open-source LLMs for real-world reasoning tasks.
[tunedailabs.com](https://tunedailabs.com)